1. **Sorting** -> ordering rows
2. **Ranking** -> relative position within data
3. **Filtering** -> selecting subsets cleanly

#### DataFrame

In [1]:
import pandas as pd

df = pd.DataFrame({
    "user_id": [101, 102, 103, 104, 105, 106],
    "age": [25, 40, 30, 40, 25, 60],
    "city": ["Pune", "Mumbai", "Delhi", "Delhi", "Pune", "Mumbai"],
    "salary": [50000, 80000, 60000, 75000, 52000, 90000]
})

### Sorting Data (`sort_values`, `sort_index`)
#### Sort by a Single Column

In [2]:
df.sort_values("salary")

,user_id,age,city,salary
0,101,25,Pune,50000
4,105,25,Pune,52000
2,103,30,Delhi,60000
3,104,40,Delhi,75000
1,102,40,Mumbai,80000
5,106,60,Mumbai,90000


In [3]:
df.sort_values("salary", ascending=False)

,user_id,age,city,salary
5,106,60,Mumbai,90000
1,102,40,Mumbai,80000
3,104,40,Delhi,75000
2,103,30,Delhi,60000
4,105,25,Pune,52000
0,101,25,Pune,50000


#### Sort by Multiple Columns

In [4]:
df.sort_values(
    by=["city", "salary"],
    ascending=[True, False]
)

,user_id,age,city,salary
3,104,40,Delhi,75000
2,103,30,Delhi,60000
5,106,60,Mumbai,90000
1,102,40,Mumbai,80000
4,105,25,Pune,52000
0,101,25,Pune,50000


Meaning:
- City -> A–Z
- Salary -> high to low *within city*

#### Sorting by Index

In [5]:
df.sort_index()

,user_id,age,city,salary
0,101,25,Pune,50000
1,102,40,Mumbai,80000
2,103,30,Delhi,60000
3,104,40,Delhi,75000
4,105,25,Pune,52000
5,106,60,Mumbai,90000


Useful after:
- Filtering
- Merging
- Concatenation

### Stable Sorting

In [6]:
df.sort_values("city", kind="mergesort")

,user_id,age,city,salary
2,103,30,Delhi,60000
3,104,40,Delhi,75000
1,102,40,Mumbai,80000
5,106,60,Mumbai,90000
0,101,25,Pune,50000
4,105,25,Pune,52000


Why this matters:
- Preserves previous order
- Important for **multi-step pipelines**

### Ranking Data (`rank`)
#### Basic Ranking

In [8]:
df["salary_rank"] = df["salary"].rank()
df

,user_id,age,city,salary,salary_rank
0,101,25,Pune,50000,1.0
1,102,40,Mumbai,80000,5.0
2,103,30,Delhi,60000,3.0
3,104,40,Delhi,75000,4.0
4,105,25,Pune,52000,2.0
5,106,60,Mumbai,90000,6.0


By default:
- Lowest value -> rank 1
- Ties -> average rank

#### Ranking Methods
When there are **ties**, the `method` parameter decides how ranks are handled.

In [9]:
df["rank_dense"] = df["salary"].rank(method="dense")
df["rank_min"] = df["salary"].rank(method="min")
df["rank_max"] = df["salary"].rank(method="max")
df

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max
0,101,25,Pune,50000,1.0,1.0,1.0,1.0
1,102,40,Mumbai,80000,5.0,5.0,5.0,5.0
2,103,30,Delhi,60000,3.0,3.0,3.0,3.0
3,104,40,Delhi,75000,4.0,4.0,4.0,4.0
4,105,25,Pune,52000,2.0,2.0,2.0,2.0
5,106,60,Mumbai,90000,6.0,6.0,6.0,6.0


| Method    | Behavior            |
| --------- | ------------------- |
| `average` | Mean of tied ranks  |
| `dense`   | No gaps             |
| `min`     | Lowest rank in tie  |
| `max`     | Highest rank in tie |


- `average` **(default)**
   - Tied values get the **mean of their rank positions**
   - Gaps may appear

- `dense`
   - Ties get the **same rank**
   - **No gaps** in ranking sequence

- `min`
   - All tied values get the **lowest rank** in the tie
   - **Gaps exist**

- `max`
   - All tied values get the **highest rank** in the tie
   - **Gaps exist**

In [11]:
df["city_salary_rank"] = (
    df
    .groupby("city")["salary"]
    .rank(ascending=False)
)
df

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
0,101,25,Pune,50000,1.0,1.0,1.0,1.0,2.0
1,102,40,Mumbai,80000,5.0,5.0,5.0,5.0,2.0
2,103,30,Delhi,60000,3.0,3.0,3.0,3.0,2.0
3,104,40,Delhi,75000,4.0,4.0,4.0,4.0,1.0
4,105,25,Pune,52000,2.0,2.0,2.0,2.0,1.0
5,106,60,Mumbai,90000,6.0,6.0,6.0,6.0,1.0


**Ranks salaries *within* each city**, giving **rank 1 to the highest salary in each city**.

1. `df.groupby("city")`
   - Splits the DataFrame into **separate groups for each city**
   - Each city is treated **independently**

2. `["salary"]`
   - Selects the salary column **inside each city group**
   - Ranking happens **per city, not globally**

### Top-N & Bottom-N Selection
#### Top values

In [12]:
df.nlargest(3, "salary")

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
5,106,60,Mumbai,90000,6.0,6.0,6.0,6.0,1.0
1,102,40,Mumbai,80000,5.0,5.0,5.0,5.0,2.0
3,104,40,Delhi,75000,4.0,4.0,4.0,4.0,1.0


#### Bottom values

In [13]:
df.nsmallest(2, "age")

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
0,101,25,Pune,50000,1.0,1.0,1.0,1.0,2.0
4,105,25,Pune,52000,2.0,2.0,2.0,2.0,1.0


### Filtering Rows
#### Boolean Filtering

In [14]:
df.loc[df["salary"] > 60000]

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
1,102,40,Mumbai,80000,5.0,5.0,5.0,5.0,2.0
3,104,40,Delhi,75000,4.0,4.0,4.0,4.0,1.0
5,106,60,Mumbai,90000,6.0,6.0,6.0,6.0,1.0


In [15]:
df.loc[
    (df["salary"] > 60000) &
    (df["city"] == "Mumbai")
]

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
1,102,40,Mumbai,80000,5.0,5.0,5.0,5.0,2.0
5,106,60,Mumbai,90000,6.0,6.0,6.0,6.0,1.0


### `.query()` - Clean Filtering Syntax

In [16]:
df.query("salary > 60000 and city == 'Mumbai'")

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
1,102,40,Mumbai,80000,5.0,5.0,5.0,5.0,2.0
5,106,60,Mumbai,90000,6.0,6.0,6.0,6.0,1.0


Variable names need `@`

In [17]:
min_salary = 60000
df.query("salary > @min_salary")

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
1,102,40,Mumbai,80000,5.0,5.0,5.0,5.0,2.0
3,104,40,Delhi,75000,4.0,4.0,4.0,4.0,1.0
5,106,60,Mumbai,90000,6.0,6.0,6.0,6.0,1.0


### Filtering with `isin()`

In [18]:
df[df["city"].isin(["Pune", "Delhi"])]

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
0,101,25,Pune,50000,1.0,1.0,1.0,1.0,2.0
2,103,30,Delhi,60000,3.0,3.0,3.0,3.0,2.0
3,104,40,Delhi,75000,4.0,4.0,4.0,4.0,1.0
4,105,25,Pune,52000,2.0,2.0,2.0,2.0,1.0


### Combining Sorting + Filtering
#### Top 2 earners per city

In [20]:
top_earner = (
    df
    .sort_values(["city", "salary"], ascending=[True, False])
    .groupby("city")
    .head(2)
)

top_earner

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
3,104,40,Delhi,75000,4.0,4.0,4.0,4.0,1.0
2,103,30,Delhi,60000,3.0,3.0,3.0,3.0,2.0
5,106,60,Mumbai,90000,6.0,6.0,6.0,6.0,1.0
1,102,40,Mumbai,80000,5.0,5.0,5.0,5.0,2.0
4,105,25,Pune,52000,2.0,2.0,2.0,2.0,1.0
0,101,25,Pune,50000,1.0,1.0,1.0,1.0,2.0


### Summary
| Task               | Best Tool          |
| ------------------ | ------------------ |
| Sort rows          | `sort_values()`    |
| Sort index         | `sort_index()`     |
| Rank values        | `rank()`           |
| Top-N rows         | `nlargest()`       |
| Complex filters    | `loc`              |
| Readable filters   | `query()`          |
| Group-wise ranking | `groupby().rank()` |


### Exercise 1
**DataFrame:**
```python
df = pd.DataFrame({
    "user_id": [101, 102, 103, 104, 105, 106, 107],
    "city": ["Pune", "Mumbai", "Delhi", "Delhi", "Pune", "Mumbai", "Pune"],
    "age": [25, 40, 30, 40, 25, 60, 35],
    "salary": [50000, 80000, 60000, 75000, 52000, 90000, 70000]
})
```
1. Sort the DataFrame by `salary` in **descending order**
2. Sort the DataFrame by `city` (A–Z) and then by `salary` (high -> low)


In [22]:
df_sorted_salary = df.sort_values("salary", ascending=False)
df_sorted_salary

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
5,106,60,Mumbai,90000,6.0,6.0,6.0,6.0,1.0
1,102,40,Mumbai,80000,5.0,5.0,5.0,5.0,2.0
3,104,40,Delhi,75000,4.0,4.0,4.0,4.0,1.0
2,103,30,Delhi,60000,3.0,3.0,3.0,3.0,2.0
4,105,25,Pune,52000,2.0,2.0,2.0,2.0,1.0
0,101,25,Pune,50000,1.0,1.0,1.0,1.0,2.0


In [23]:
df_sorted_city_salary = df.sort_values(
    by=["city", "salary"],
    ascending=[True, False]
)

df_sorted_city_salary

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
3,104,40,Delhi,75000,4.0,4.0,4.0,4.0,1.0
2,103,30,Delhi,60000,3.0,3.0,3.0,3.0,2.0
5,106,60,Mumbai,90000,6.0,6.0,6.0,6.0,1.0
1,102,40,Mumbai,80000,5.0,5.0,5.0,5.0,2.0
4,105,25,Pune,52000,2.0,2.0,2.0,2.0,1.0
0,101,25,Pune,50000,1.0,1.0,1.0,1.0,2.0


### Exercise 2
1. Create a column `salary_rank` (overall rank, highest salary = rank 1)
2. Create a column `city_salary_rank` (rank salary **within each city**, highest = rank 1)

In [25]:
df["salary_rank"] = df["salary"].rank(
    ascending=False,
    method="dense"
)
df

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
0,101,25,Pune,50000,6.0,1.0,1.0,1.0,2.0
1,102,40,Mumbai,80000,2.0,5.0,5.0,5.0,2.0
2,103,30,Delhi,60000,4.0,3.0,3.0,3.0,2.0
3,104,40,Delhi,75000,3.0,4.0,4.0,4.0,1.0
4,105,25,Pune,52000,5.0,2.0,2.0,2.0,1.0
5,106,60,Mumbai,90000,1.0,6.0,6.0,6.0,1.0


In [26]:
df["city_salary_rank"] = (
    df
    .groupby("city")["salary"]
    .rank(ascending=False, method="dense")
)
df

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
0,101,25,Pune,50000,6.0,1.0,1.0,1.0,2.0
1,102,40,Mumbai,80000,2.0,5.0,5.0,5.0,2.0
2,103,30,Delhi,60000,4.0,3.0,3.0,3.0,2.0
3,104,40,Delhi,75000,3.0,4.0,4.0,4.0,1.0
4,105,25,Pune,52000,5.0,2.0,2.0,2.0,1.0
5,106,60,Mumbai,90000,1.0,6.0,6.0,6.0,1.0


### Exercise 3
1. Get the **top 3 highest-paid users**
2. Get the **2 youngest users**

In [27]:
top_3_earners = df.nlargest(3, "salary")
top_3_earners

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
5,106,60,Mumbai,90000,1.0,6.0,6.0,6.0,1.0
1,102,40,Mumbai,80000,2.0,5.0,5.0,5.0,2.0
3,104,40,Delhi,75000,3.0,4.0,4.0,4.0,1.0


In [28]:
youngest_2 = df.nsmallest(2, "age")
youngest_2

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
0,101,25,Pune,50000,6.0,1.0,1.0,1.0,2.0
4,105,25,Pune,52000,5.0,2.0,2.0,2.0,1.0


### Exercise 4
1. Filter users with salary **greater than 60000**
2. Filter users from **Mumbai** with salary **greater than 60000**
3. Filter users whose city is **either Pune or Delhi**

In [29]:
high_salary_users = df.loc[df["salary"] > 60000]
high_salary_users

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
1,102,40,Mumbai,80000,2.0,5.0,5.0,5.0,2.0
3,104,40,Delhi,75000,3.0,4.0,4.0,4.0,1.0
5,106,60,Mumbai,90000,1.0,6.0,6.0,6.0,1.0


In [30]:
mumbai_high_salary = df.loc[
    (df["city"] == "Mumbai") &
    (df["salary"] > 60000)
]
mumbai_high_salary

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
1,102,40,Mumbai,80000,2.0,5.0,5.0,5.0,2.0
5,106,60,Mumbai,90000,1.0,6.0,6.0,6.0,1.0


In [31]:
pune_delhi_users = df.loc[
    df["city"].isin(["Pune", "Delhi"])
]
pune_delhi_users

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
0,101,25,Pune,50000,6.0,1.0,1.0,1.0,2.0
2,103,30,Delhi,60000,4.0,3.0,3.0,3.0,2.0
3,104,40,Delhi,75000,3.0,4.0,4.0,4.0,1.0
4,105,25,Pune,52000,5.0,2.0,2.0,2.0,1.0


### Exercise 5
Re-write **Exercise 4.2** using `.query()`.

In [32]:
mumbai_high_salary_q = df.query(
    "city == 'Mumbai' and salary > 60000"
)
mumbai_high_salary_q

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
1,102,40,Mumbai,80000,2.0,5.0,5.0,5.0,2.0
5,106,60,Mumbai,90000,1.0,6.0,6.0,6.0,1.0


### Exercise 6
For each city, show the **top 2 earners**, sorted by city name.

In [33]:
top_2_per_city = (
    df
    .sort_values(["city", "salary"], ascending=[True, False])
    .groupby("city")
    .head(2)
)
top_2_per_city

,user_id,age,city,salary,salary_rank,rank_dense,rank_min,rank_max,city_salary_rank
3,104,40,Delhi,75000,3.0,4.0,4.0,4.0,1.0
2,103,30,Delhi,60000,4.0,3.0,3.0,3.0,2.0
5,106,60,Mumbai,90000,1.0,6.0,6.0,6.0,1.0
1,102,40,Mumbai,80000,2.0,5.0,5.0,5.0,2.0
4,105,25,Pune,52000,5.0,2.0,2.0,2.0,1.0
0,101,25,Pune,50000,6.0,1.0,1.0,1.0,2.0
